
# Brain Tumor Classification (MRI) - Trained on Real Dataset

This notebook trains standard ML models (Logistic Regression, SVM, Random Forest) and optional DL models (CNN, ResNet50) on the **Brain Tumor MRI Dataset**.



## Workflow
1. Locate dataset automatically (local + Kaggle paths)
2. Load MRI images with true labels
3. Train/Test split (or use provided Testing folder)
4. Train ML models and evaluate
5. Show confusion matrices and sample predictions
6. (Optional) Train CNN + ResNet50 if TensorFlow is available


In [ ]:

import os
from pathlib import Path
import random
import warnings
import numpy as np

# avoid cache-permission warnings in restricted environments
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl_cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/.cache")
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)
os.makedirs(os.path.join(os.environ["XDG_CACHE_HOME"], "fontconfig"), exist_ok=True)

import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (6, 5)


In [ ]:

# ---------- Dataset Discovery ----------

# Handles running notebook from either:
# 1) brain tumor classification/ folder
# 2) project root folder

BASE_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "brain tumor classification",
    Path("."),
]


def _dataset_roots():
    rels = [
        Path("dataset/brain-tumor-mri-dataset"),
        Path("dataset/Brain-Tumor-MRI-Dataset"),
        Path("dataset/archive"),
        Path("dataset"),
    ]
    out = []
    for b in BASE_DIR_CANDIDATES:
        for r in rels:
            out.append((b / r).resolve())
    out += [
        Path("/kaggle/input/brain-tumor-mri-dataset"),
        Path("/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"),
    ]
    # unique order-preserving
    uniq = []
    seen = set()
    for x in out:
        sx = str(x)
        if sx not in seen:
            seen.add(sx)
            uniq.append(x)
    return uniq


def _has_train_split(folder: Path) -> bool:
    for d in ["Training", "Train", "training", "train"]:
        if (folder / d).exists():
            return True
    return False


def _has_class_folders(folder: Path) -> bool:
    class_names = {"glioma", "meningioma", "pituitary", "notumor", "no_tumor", "no-tumor"}
    dirs = [p.name.lower().replace(" ", "_") for p in folder.iterdir() if p.is_dir()]
    return any(name in class_names for name in dirs)


def find_dataset_root() -> Path:
    for c in _dataset_roots():
        if not c.exists():
            continue

        # direct match
        if _has_train_split(c) or _has_class_folders(c):
            return c

        # one-level nested match (handles dataset/archive/...)
        for child in [p for p in c.iterdir() if p.is_dir()]:
            if _has_train_split(child) or _has_class_folders(child):
                return child

    raise FileNotFoundError(
        "Dataset not found. Put dataset inside 'brain tumor classification/dataset/' or run in Kaggle with dataset attached."
    )


def pick_split_dirs(root: Path):
    train_candidates = ["Training", "Train", "training", "train"]
    test_candidates = ["Testing", "Test", "testing", "test"]

    train_dir = next((root / d for d in train_candidates if (root / d).exists()), None)
    test_dir = next((root / d for d in test_candidates if (root / d).exists()), None)

    # fallback: root itself has class folders
    if train_dir is None:
        train_dir = root

    return train_dir, test_dir


def normalize_label(name: str) -> str:
    x = name.lower().strip().replace("-", "_").replace(" ", "_")
    if x in {"notumor", "no_tumor", "no-tumor", "normal"}:
        return "no_tumor"
    if "glioma" in x:
        return "glioma"
    if "meningioma" in x:
        return "meningioma"
    if "pituitary" in x:
        return "pituitary"
    return x


from typing import Optional

def _unwrap_nested_split_dir(split_dir: Path) -> Path:
    """If split dir is nested like Training/Training/<class>, descend automatically."""
    for _ in range(3):
        dirs = [p for p in split_dir.iterdir() if p.is_dir()]
        if not dirs:
            break
        names = {normalize_label(d.name) for d in dirs}
        if any(n in {"glioma", "meningioma", "pituitary", "no_tumor"} for n in names):
            return split_dir
        if len(dirs) == 1:
            split_dir = dirs[0]
            continue
        break
    return split_dir


def list_images(split_dir: Path, max_per_class: Optional[int] = None):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    items = []

    split_dir = _unwrap_nested_split_dir(split_dir)
    class_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])
    if not class_dirs:
        raise ValueError(f"No class folders found in: {split_dir}")

    for cdir in class_dirs:
        label = normalize_label(cdir.name)
        count = 0
        for p in sorted(cdir.rglob("*")):
            if p.is_file() and p.suffix.lower() in exts:
                items.append((p, label))
                count += 1
                if max_per_class is not None and count >= max_per_class:
                    break
    return items


DATASET_ROOT = find_dataset_root()
TRAIN_DIR, TEST_DIR = pick_split_dirs(DATASET_ROOT)

print("Dataset root:", DATASET_ROOT)
print("Train dir:", TRAIN_DIR)
print("Test dir:", TEST_DIR if TEST_DIR else "<not provided, will split train set>")


In [ ]:

# ---------- Build train/test item lists ----------
MAX_PER_CLASS_TRAIN = 1200   # adjust if RAM is low
MAX_PER_CLASS_TEST = 400

train_items = list_images(TRAIN_DIR, max_per_class=MAX_PER_CLASS_TRAIN)

if TEST_DIR is not None and TEST_DIR.exists():
    test_items = list_images(TEST_DIR, max_per_class=MAX_PER_CLASS_TEST)
else:
    # if no explicit test folder, create split from training images
    labels = [lbl for _, lbl in train_items]
    train_items, test_items = train_test_split(
        train_items,
        test_size=0.2,
        random_state=SEED,
        stratify=labels,
    )

print(f"Train images: {len(train_items)}")
print(f"Test images:  {len(test_items)}")

from collections import Counter
train_dist = dict(Counter([y for _, y in train_items]))
test_dist = dict(Counter([y for _, y in test_items]))
print("Train class distribution:", train_dist)
print("Test class distribution:", test_dist)

if len(train_dist) < 2:
    raise ValueError(
        f"Training data has only one class: {list(train_dist.keys())}. "
        "Check dataset folder structure. Expected class folders like glioma/meningioma/pituitary/notumor."
    )
if len(test_dist) < 2:
    raise ValueError(
        f"Testing data has only one class: {list(test_dist.keys())}. "
        "Check Testing folder structure or use train_test_split from full training set."
    )


In [ ]:

# ---------- Load grayscale features for ML ----------

ML_IMAGE_SIZE = 96


def load_flattened(items, image_size=96):
    X, y = [], []
    for path, label in items:
        img = Image.open(path).convert("L").resize((image_size, image_size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        X.append(arr.flatten())
        y.append(label)
    return np.asarray(X, dtype=np.float32), np.asarray(y)


X_train_raw, y_train_text = load_flattened(train_items, ML_IMAGE_SIZE)
X_test_raw, y_test_text = load_flattened(test_items, ML_IMAGE_SIZE)

le = LabelEncoder()
y_train = le.fit_transform(y_train_text)
y_test = le.transform(y_test_text)

print("Classes:", list(le.classes_))
print("X_train:", X_train_raw.shape, "X_test:", X_test_raw.shape)


In [ ]:

# ---------- Scale + PCA ----------

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_raw)
X_test_s = scaler.transform(X_test_raw)

n_components = min(180, X_train_s.shape[1], X_train_s.shape[0] - 1)
pca = PCA(n_components=n_components, random_state=SEED)
X_train_p = pca.fit_transform(X_train_s)
X_test_p = pca.transform(X_test_s)

print("PCA components:", n_components)
print("Explained variance ratio:", round(float(np.sum(pca.explained_variance_ratio_)), 4))


In [ ]:

# ---------- Train and evaluate standard ML models ----------

unique_train = np.unique(y_train)
if len(unique_train) < 2:
    readable = [str(x) for x in le.inverse_transform(unique_train)] if len(unique_train) else []
    raise ValueError(
        f"Training labels contain only {len(unique_train)} class: {readable}. "
        "Fix dataset path/split and rerun all cells from the top."
    )

models = {
    "logistic_regression": LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        random_state=SEED,
    ),
    "svm_rbf": SVC(
        kernel="rbf",
        C=4.0,
        gamma="scale",
        probability=True,
        random_state=SEED,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
}

ml_results = {}

for name, model in models.items():
    model.fit(X_train_p, y_train)
    pred = model.predict(X_test_p)
    acc = accuracy_score(y_test, pred)
    ml_results[name] = {"model": model, "accuracy": acc, "pred": pred}

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, pred, target_names=le.classes_, zero_division=0))

    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(cmap="Blues", xticks_rotation=30)
    plt.title(f"{name} - Confusion Matrix")
    plt.show()

best_ml = max(ml_results.items(), key=lambda kv: kv[1]["accuracy"])
print(f"Best ML model: {best_ml[0]} ({best_ml[1]['accuracy']:.4f})")


In [ ]:

# ---------- Real sample prediction demo ----------

best_model_name, best_pack = best_ml
best_model = best_pack["model"]

idx = np.random.randint(0, len(test_items))
sample_path, true_label = test_items[idx]

sample_img = Image.open(sample_path).convert("L").resize((ML_IMAGE_SIZE, ML_IMAGE_SIZE))
sample_arr = np.asarray(sample_img, dtype=np.float32).flatten()[None, :] / 255.0
sample_arr = scaler.transform(sample_arr)
sample_arr = pca.transform(sample_arr)

pred_idx = best_model.predict(sample_arr)[0]
pred_label = le.inverse_transform([pred_idx])[0]

print("Sample path:", sample_path)
print("True label:", true_label)
print("Predicted label:", pred_label)

plt.figure(figsize=(4, 4))
plt.imshow(Image.open(sample_path).convert("RGB"))
plt.axis("off")
plt.title(f"True: {true_label} | Pred: {pred_label}")
plt.show()


In [ ]:

# ---------- Optional DL section (CNN + ResNet50) ----------

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
    from tensorflow.keras.models import Model
    from tensorflow.keras.applications import ResNet50
except Exception as e:
    tf = None
    print("TensorFlow not available:", e)

RUN_DEEP = tf is not None
print("RUN_DEEP:", RUN_DEEP)


In [ ]:

if RUN_DEEP:
    DL_IMAGE_SIZE = 128
    MAX_PER_CLASS_DL = 350  # keep moderate for local runtime

    train_items_dl = list_images(TRAIN_DIR, max_per_class=MAX_PER_CLASS_DL)
    if TEST_DIR is not None and TEST_DIR.exists():
        test_items_dl = list_images(TEST_DIR, max_per_class=min(150, MAX_PER_CLASS_DL // 2))
    else:
        labels_dl = [lbl for _, lbl in train_items_dl]
        train_items_dl, test_items_dl = train_test_split(
            train_items_dl,
            test_size=0.2,
            random_state=SEED,
            stratify=labels_dl,
        )

    def load_rgb(items, image_size=128):
        Xc, yc = [], []
        for path, label in items:
            img = Image.open(path).convert("RGB").resize((image_size, image_size))
            Xc.append(np.asarray(img, dtype=np.float32) / 255.0)
            yc.append(label)
        return np.asarray(Xc, dtype=np.float32), np.asarray(yc)

    Xc_train, yc_train_text = load_rgb(train_items_dl, DL_IMAGE_SIZE)
    Xc_test, yc_test_text = load_rgb(test_items_dl, DL_IMAGE_SIZE)

    le_dl = LabelEncoder()
    yc_train = le_dl.fit_transform(yc_train_text)
    yc_test = le_dl.transform(yc_test_text)

    n_classes = len(le_dl.classes_)
    yc_train_oh = tf.keras.utils.to_categorical(yc_train, n_classes)
    yc_test_oh = tf.keras.utils.to_categorical(yc_test, n_classes)

    print("DL classes:", list(le_dl.classes_))
    print("Xc_train:", Xc_train.shape, "Xc_test:", Xc_test.shape)

    # CNN
    cnn = Sequential([
        Conv2D(32, (3, 3), activation="relu", input_shape=(DL_IMAGE_SIZE, DL_IMAGE_SIZE, 3)),
        MaxPooling2D(2),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(2),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D(2),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(n_classes, activation="softmax"),
    ])
    cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    cnn.fit(Xc_train, yc_train_oh, epochs=4, batch_size=32, validation_split=0.15, verbose=1)
    _, cnn_acc = cnn.evaluate(Xc_test, yc_test_oh, verbose=0)
    print("CNN accuracy:", round(float(cnn_acc), 4))

    # ResNet50
    base = ResNet50(include_top=False, weights="imagenet", input_shape=(DL_IMAGE_SIZE, DL_IMAGE_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.3)(x)
    out = Dense(n_classes, activation="softmax")(x)
    resnet = Model(inputs=base.input, outputs=out)
    resnet.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    resnet.fit(Xc_train, yc_train_oh, epochs=3, batch_size=24, validation_split=0.15, verbose=1)
    _, resnet_acc = resnet.evaluate(Xc_test, yc_test_oh, verbose=0)
    print("ResNet50 accuracy:", round(float(resnet_acc), 4))

    # one deep sample prediction
    j = np.random.randint(0, len(test_items_dl))
    pth, true_lbl = test_items_dl[j]
    arr = np.asarray(Image.open(pth).convert("RGB").resize((DL_IMAGE_SIZE, DL_IMAGE_SIZE)), dtype=np.float32)[None, ...] / 255.0
    pr = resnet.predict(arr, verbose=0)[0]
    pred_lbl = le_dl.classes_[int(np.argmax(pr))]

    print("Deep sample path:", pth)
    print("True:", true_lbl, "| Pred (ResNet50):", pred_lbl)

    plt.figure(figsize=(4, 4))
    plt.imshow(Image.open(pth).convert("RGB"))
    plt.axis("off")
    plt.title(f"Deep True: {true_lbl} | Pred: {pred_lbl}")
    plt.show()
else:
    print("Skipped deep learning models. Install TensorFlow to run CNN/ResNet50 cells.")
